In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException, WebDriverException
from webdriver_manager.chrome import ChromeDriverManager
import re
import time
import pandas as pd
import os
import traceback
import numpy as np
import textdistance
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from unidecode import unidecode


In [2]:
wine_df = pd.read_csv('../data/cleaned_wine_df.csv')

In [3]:
# drop row that was named twice in dm df to avoid issue with vivino scraping
wine_df = wine_df.drop(2313).reset_index(drop=True)

In [4]:
## drop cleanskin wines
wine_df = wine_df[~wine_df["product_brand"].str.contains("cleanskin", case=False)].reset_index(drop=True)

In [5]:
## build full product names to search in vivino
dm_product_hrefs = list(wine_df.href.values)
dm_brand_names = list(wine_df.product_brand.values)
dm_product_names = list(wine_df.product_name.values)
dm_brand_product_names = []
for _, row in wine_df.iterrows():
    dm_brand_product_names.append(f"{row.product_brand} {row.product_name}")

In [6]:
# Create options object
options = Options()

#options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-infobars')
options.add_argument('--disable-extensions')
options.add_argument('--disable-notifications')
options.add_argument('--window-size=1724,968')
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/96.0.4664.110 Safari/537.36"
options.add_argument(f'user-agent={user_agent}')

In [7]:
def calculate_jaccard_similarity(dm_string, vivino_string):
    stop_words = set(stopwords.words('english'))
    
    # Tokenize the product names into words and convert to lower case
    dm_tokens = word_tokenize(unidecode(dm_string.lower()))
    vivino_tokens = word_tokenize(unidecode(vivino_string.lower()))
    
    # Remove stop words and non-alphanumeric tokens
    dm_tokens = [word for word in dm_tokens if word.isalnum() and word not in stop_words]
    vivino_tokens = [word for word in vivino_tokens if word.isalnum() and word not in stop_words]
    
    # Convert lists to sets
    dm_set = set(dm_tokens)
    vivino_set = set(vivino_tokens)
    
    # Calculate Jaccard similarity
    intersection = dm_set.intersection(vivino_set)
    union = dm_set.union(vivino_set)
    
    if not union:
        return 0.0  # Handle the case when both sets are empty
    
    jaccard_similarity = len(intersection) / len(union)
    
    return jaccard_similarity


def get_most_similar_wine_card(wine_cards, dm_full_product_name):
    similarities = []
    max_similarity = 0
    max_similarity_i = 0
    for i in range(len(wine_cards)):
        product_link = wine_cards[i].find_element(By.CLASS_NAME, "link-color-alt-grey")
        vivino_product_name = product_link.text
        #similarity = textdistance.levenshtein.normalized_similarity(dm_full_product_name, vivino_product_name)
        similarity = calculate_jaccard_similarity(dm_full_product_name, vivino_product_name)
        similarities.append(similarity)
        if similarity > max_similarity:
            max_similarity = similarity
            max_similarity_i = i
    print(f"Max similarity found: {max_similarity}")
    print("................................")
    print(f"Name of dm product: {dm_full_product_name}")
    first_product_name = wine_cards[0].find_element(By.CLASS_NAME, "link-color-alt-grey").text
    most_similar_product_name = wine_cards[max_similarity_i].find_element(By.CLASS_NAME, "link-color-alt-grey").text
    print(f"Name of first product: {first_product_name}")
    print(f"Name of returned product: {most_similar_product_name}")
    print("................................")
    return wine_cards[max_similarity_i]

def remove_punctuation(text):
    return re.sub(r'[^\w\s]', '', text)

def remove_spaces(text):
    return re.sub(r'\s+', '', text)

def calculate_brand_name_presence(dm_brand_name, vivino_product_name):
    dm_brand_name = remove_spaces(remove_punctuation(unidecode(dm_brand_name.lower())))
    vivino_product_name = remove_spaces(remove_punctuation(unidecode(vivino_product_name.lower())))
    if dm_brand_name in vivino_product_name:
        return True
    else:
        return False

def restart_loop_prep(message, loops, max_loops=50):
    print(message)
    print("attempted loops:", loops)
    if loops > max_loops:
        raise Exception("!!!Max loops. Stopping scrapping")
    time.sleep(1)
    loops += 1
    return loops

total_wines = len(dm_brand_product_names)
max_loops = 50
starting_i = 6040
similarity_threshold = 0.5

previous_search_results_text = np.nan

# setup driver
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
page = "https://www.vivino.com/AU/en/"

#while starting_i < total_wines-1:

# lists to make into dataframe
dm_brand_names_subset = []
dm_product_names_subset = []
dm_product_hrefs_subset = []
vivino_product_names = []
vivino_hrefs = []
    
try:
    try:
        driver.get(page)
    except WebDriverException:
        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
        driver.get(page)

    
    # Set maximum wait time for elements to load
    wait = WebDriverWait(driver, 10)  # Adjust the timeout value as needed

    ## loop through and scrape products 
    for i in range(starting_i, total_wines, 1):

        # get product name to query
        query = dm_brand_product_names[i]
        print(f"Searching vivino for i={i} of {total_wines-1}: {query}")
    
        # add query to search bar and search
        #time.sleep(1)
        search_bar_content = np.nan
        loops = 0
        while True:
            try:
                search_bar = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.CLASS_NAME, "searchBar_searchInput__Jje-K")))
                search_bar.clear()
                search_bar.send_keys(query)
                search_bar_content = search_bar.get_attribute("value")
            except StaleElementReferenceException:
                loops = restart_loop_prep("StaleElementReferenceException while updating search bar. Retrying...", loops)
                continue
            if search_bar_content != query:
                loops = restart_loop_prep("Query not added to search bar. Retrying...", loops)
                continue
            break
        search_bar.send_keys(Keys.RETURN)
        
        # check search results have updated
        #time.sleep(1)
        loops = 0
        results_found = False
        while True:
            try:
                search_results_list = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "search-results-list")))
                search_results_text = search_results_list.text
                if search_results_text == previous_search_results_text:
                    loops = restart_loop_prep("Failed to load new search results list. Wait and try again...", loops)
                    continue
                results_found = True
                #results_found = True
            except TimeoutException:
                print("!!!No search results found")
            except StaleElementReferenceException:
                loops = restart_loop_prep("StaleElementReferenceException while trying to check results list. Wait and try again...", loops)
                continue
            break
        previous_search_results_text = search_results_text

        #get href and name of product
        #time.sleep(1)
        loops = 0
        while True:
            try:
                if results_found:
                    wine_cards = driver.find_elements(By.CLASS_NAME, "default-wine-card")
                    product_link = wine_cards[0].find_element(By.CLASS_NAME, "link-color-alt-grey")

                    # find card with closest match if first card isn't similar or brand isn't in name
                    vivino_product_name = product_link.text
                    dm_brand_name = dm_brand_names[i]
                    dm_brand_product_name = dm_brand_product_names[i]

                    brand_name_present = calculate_brand_name_presence(dm_brand_name, vivino_product_name)
                    #similarity = textdistance.levenshtein.normalized_similarity(dm_brand_product_name, vivino_product_name)
                    similarity = calculate_jaccard_similarity(dm_brand_product_name, vivino_product_name)
                    poor_result = False
                    if not brand_name_present:
                        print("Brand name not present. Searching most for most similar result...")
                        poor_result = True
                    if not poor_result and similarity < similarity_threshold:
                        print(f"First search result only rated {similarity} for similarity. Searching for most similar result...")
                        poor_result = True
                    if poor_result:
                        better_wine_card = get_most_similar_wine_card(wine_cards, dm_brand_product_name)
                        product_link = better_wine_card.find_element(By.CLASS_NAME, "link-color-alt-grey")
                        vivino_product_name = product_link.text
                    
                    vivino_product_href = product_link.get_attribute("href")
                else:
                    vivino_product_name = ""
                    vivino_product_href = ""
            except StaleElementReferenceException:
                loops = restart_loop_prep("StaleElementReferenceException while getting product details. Retrying...", loops)
                continue
            break

        # populate lists
        dm_brand_names_subset.append(dm_brand_names[i])
        dm_product_names_subset.append(dm_product_names[i])
        dm_product_hrefs_subset.append(dm_product_hrefs[i])
        vivino_product_names.append(vivino_product_name)
        vivino_hrefs.append(vivino_product_href)
        print(f"Finished scraping for i = {i} of {total_wines-1}\n")

except Exception as e:
    print(f"Failed scraping at i = {i}")
    print("saving progress")
    traceback.print_exc()
    print(e)


finally:

    #save results
    vivino_href_df = {
        "dm_brand_name" : dm_brand_names_subset,
        "dm_product_name" : dm_product_names_subset,
        "dm_href" : dm_product_hrefs_subset,
        "vivino_product_name" : vivino_product_names,
        "vivino_href" : vivino_hrefs,
    }
    vivino_href_df = pd.DataFrame(vivino_href_df)

    # save df to csv
    file_path = f"../data/vivino/similarity_href_subsets/vivino_hrefs_dataset_i_{starting_i}-{i-1}.csv"
    #file_path = f"../data/vivino/similarity_href_subsets/vivino_hrefs_dataset_missing1.csv" # backtracking for misssing product
    vivino_href_df.to_csv(file_path, index=False)

    starting_i = i

    time.sleep(3)

# Close the webpage
driver.quit()

Searching vivino for i=6040 of 6054: Château Ausone St-Émilion 1er Grand Cru 'A' 2012
Brand name not present. Searching most for most similar result...
Max similarity found: 0.7142857142857143
................................
Name of dm product: Château Ausone St-Émilion 1er Grand Cru 'A' 2012
Name of first product: Chateau Pierre 1er (Chateau Croix Figeac) Saint-Émilion Grand Cru 2012
Name of returned product: Château Ausone Chapelle D'Ausone Saint-Émilion Grand Cru 2012
................................
Finished scraping for i = 6040 of 6054

Searching vivino for i=6041 of 6054: Emilio Moro Malleolus
Failed to load new search results list. Wait and try again...
attempted loops: 0
Finished scraping for i = 6041 of 6054

Searching vivino for i=6042 of 6054: Penfolds Grange 2015
Failed to load new search results list. Wait and try again...
attempted loops: 0
Finished scraping for i = 6042 of 6054

Searching vivino for i=6043 of 6054: Ten Minutes By Tractor 10X Rosé
Query not added to sea

In [8]:
vivino_href_df

,dm_brand_name,dm_product_name,dm_href,vivino_product_name,vivino_href
0,Château Ausone,St-Émilion 1er Grand Cru 'A' 2012,https://www.danmurphys.com.au/product/DM_56697...,Château Ausone Chapelle D'Ausone Saint-Émilion...,https://www.vivino.com/AU/en/wines/10072527
1,Emilio Moro,Malleolus,https://www.danmurphys.com.au/product/DM_56624...,Emilio Moro Malleolus,https://www.vivino.com/AU/en/wines/1731128
2,Penfolds,Grange 2015,https://www.danmurphys.com.au/product/DM_56043...,Penfolds Grange 2015,https://www.vivino.com/AU/en/wines/14141861
3,Ten Minutes By Tractor,10X Rosé,https://www.danmurphys.com.au/product/DM_56025...,Penfolds Grange 2015,https://www.vivino.com/AU/en/wines/14141861
4,Penfolds,Bin 138 Shiraz Grenache Mataro 2017,https://www.danmurphys.com.au/product/DM_56020...,Ten Minutes By Tractor 10X Rosé,https://www.vivino.com/AU/en/wines/4886720
5,Kaesler Wines,Old Bastard Shiraz 2014 750mL,https://www.danmurphys.com.au/product/DM_55877...,Kaesler Old Bastard Shiraz,https://www.vivino.com/AU/en/wines/4718643
6,Torbreck,RunRig Shiraz 2016,https://www.danmurphys.com.au/product/DM_55034...,Torbreck Woodcutter'S Shiraz 2016,https://www.vivino.com/AU/en/wines/86701938
7,Woodvale,Orchard Road Cabernet Sauvignon 2021,https://www.danmurphys.com.au/product/DM_54990...,Woodvale Orchard Road Cabernet Sauvignon 2021,https://www.vivino.com/AU/en/wines/172541163
8,Tyrrell's,Winemakers Vat 9 Shiraz 1.5L,https://www.danmurphys.com.au/product/DM_54746...,Tyrrell'S Vat 9 Shiraz,https://www.vivino.com/AU/en/wines/3725997
9,Greenhough,Hope Vineyard Pinot Noir,https://www.danmurphys.com.au/product/DM_54328...,Greenhough Hope Vineyard Pinot Noir,https://www.vivino.com/AU/en/wines/4886451


In [9]:
"""
#save results
vivino_href_df = {
    "dm_brand_name" : dm_brand_names_subset,
    "dm_product_name" : dm_product_names_subset,
    "dm_href" : dm_product_hrefs_subset,
    "vivino_product_name" : vivino_product_names,
    "vivino_href" : vivino_hrefs,
}
vivino_href_df = pd.DataFrame(vivino_href_df)

# save df to csv
file_path = f"../data/vivino/similarity_href_subsets/vivino_hrefs_dataset_i_{starting_i}-{i-1}.csv"
vivino_href_df.to_csv(file_path, index=False)
"""

'\n#save results\nvivino_href_df = {\n    "dm_brand_name" : dm_brand_names_subset,\n    "dm_product_name" : dm_product_names_subset,\n    "dm_href" : dm_product_hrefs_subset,\n    "vivino_product_name" : vivino_product_names,\n    "vivino_href" : vivino_hrefs,\n}\nvivino_href_df = pd.DataFrame(vivino_href_df)\n\n# save df to csv\nfile_path = f"../data/vivino/similarity_href_subsets/vivino_hrefs_dataset_i_{starting_i}-{i-1}.csv"\nvivino_href_df.to_csv(file_path, index=False)\n'